In [21]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [22]:
# Hämta datasetet
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olaflundstrom/nordcauc-urban-mobility-intelligence",
    "traffic_flow_sensors.csv"
)

/var/folders/16/f9ng7c413pqgml7b29mwnvjh0000gn/T/ipykernel_36144/4129057985.py:2: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


ConnectionError: HTTPSConnectionPool(host='api.kaggle.com', port=443): Max retries exceeded with url: /v1/datasets.DatasetApiService/GetDataset (Caused by NameResolutionError("HTTPSConnection(host='api.kaggle.com', port=443): Failed to resolve 'api.kaggle.com' ([Errno 8] nodename nor servname provided, or not known)"))

In [ ]:
# Behåll endast observationer med god datakvalitet
df_processed = df[df["data_quality_flag"] == "good"].copy()

# Undersök fördelningen av target
print(df_processed["congestion_level"].value_counts())
print(df_processed["congestion_level"].value_counts(normalize=True))

# Ta bort kolumner som inte ska användas
columns_to_drop = [
    "sensor_id", "city", "country", "district",
    "timestamp", "date", "latitude", "longitude",
    "ev_count_15min", "ai_predicted_volume",
    "ai_predicted_congestion", "ai_model_version",
    "data_quality_flag"
]

df_processed = df_processed.drop(columns=columns_to_drop)

In [ ]:
# Dela upp features och target
X = df_processed.drop(columns=["congestion_level"])
y = df_processed["congestion_level"]

# Dela upp träningsdata och testdata
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Skapa baseline
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Utvärdera
y_pred = baseline.predict(X_test)

print("Baseline accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

In [23]:
# Identifiera numeriska och kategoriska features
numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns
# Hantera saknade numeriska värden
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Hantera kategoriska värden
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Kombinera preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Skapa Decision Tree
tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

# Träna modellen
tree_model.fit(X_train, y_train)

# Förutsäg resultat
tree_predictions = tree_model.predict(X_test)

# Utvärdera
print("Decision Tree accuracy:",
      accuracy_score(y_test, tree_predictions))

print("Decision Tree F1:",
      f1_score(y_test, tree_predictions, average="macro"))

print(classification_report(y_test, tree_predictions))

print("Confusion matrix:")
print(confusion_matrix(y_test, tree_predictions))


/var/folders/16/f9ng7c413pqgml7b29mwnvjh0000gn/T/ipykernel_36144/2794857052.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


Decision Tree accuracy: 0.9999728297785627
Decision Tree F1: 0.9999611248015937
              precision    recall  f1-score   support

   free_flow       1.00      1.00      1.00     17007
       heavy       1.00      1.00      1.00      5830
       light       1.00      1.00      1.00      6798
    moderate       1.00      1.00      1.00      7170

    accuracy                           1.00     36805
   macro avg       1.00      1.00      1.00     36805
weighted avg       1.00      1.00      1.00     36805

Confusion matrix:
[[17007     0     0     0]
 [    0  5829     0     1]
 [    0     0  6798     0]
 [    0     0     0  7170]]
